# Дружелюбный построитель тепловых карт

Этот ноутбук автоматически скачивает свежий `heatmap.py` из GitHub, показывает синтетические примеры таблиц и строит heatmap по вашим Excel/CSV файлам без правки Python-кода.

In [ ]:
%pip -q install pandas numpy matplotlib openpyxl

## 1. Подготовьте код и загрузите свои данные

`heatmap.py` и синтетические примеры из папки `examples/` скачиваются автоматически из публичного репозитория. Загружать вручную нужно только ваши Excel/CSV файлы с экспериментами.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

RAW_BASE_URL = "https://raw.githubusercontent.com/KIT-MAKSGOR-Vaillka/heatmap/main"
RAW_HEATMAP_URL = f"{RAW_BASE_URL}/heatmap.py"
urlretrieve(RAW_HEATMAP_URL, "heatmap.py")
print("Скачан свежий heatmap.py из GitHub:", RAW_HEATMAP_URL)

Path("examples").mkdir(exist_ok=True)
example_files = [
    "example_triplet_pH_6_5.xlsx",
    "example_triplet_pH_7_4.xlsx",
    "example_long.csv",
    "example_wide_dose.csv",
]
for example_name in example_files:
    target = Path("examples") / example_name
    urlretrieve(f"{RAW_BASE_URL}/examples/{example_name}", target)
print("Скачаны синтетические примеры из папки examples/.")

try:
    from google.colab import files
    uploaded = files.upload()
    print("Загружено пользовательских файлов:", ", ".join(uploaded.keys()) or "ничего")
except Exception:
    print("Кнопка загрузки работает в Google Colab. При локальном запуске положите файлы рядом с ноутбуком.")

print("\nФайлы в текущей среде:")
for local_path in sorted(Path(".").glob("*")):
    if local_path.is_file():
        print(" -", local_path.name)

print("\nПримеры в папке examples/:")
for local_path in sorted(Path("examples").glob("*")):
    if local_path.is_file():
        print(" -", local_path)


## 2. Настройте данные

Ниже находится главная форма настройки. Для обычной работы с Excel меняются только имена файлов, pH и правила группировки образцов.

### Если у вас основной Excel-формат

Используйте режим `Excel-файлы списком file=pH`, если у вас один или несколько Excel-файлов, где:

- первый столбец - доза, например `Dose, Gy`;
- дальше каждый образец занимает 3 столбца подряд: `mean`, `SD`, `N`;
- концентрация написана в названии образца в скобках: `NanoA 5% (0.005)`;
- каждый Excel-файл соответствует одному pH.

В поле `excel_files_with_ph` пишите пары `имя_файла=pH` через точку с запятой.

Пример для двух Excel-файлов:

```text
examples/example_triplet_pH_6_5.xlsx=6.5; examples/example_triplet_pH_7_4.xlsx=7.4
```

Пример для трех pH:

```text
experiment_pH_6_0.xlsx=6.0; experiment_pH_6_5.xlsx=6.5; experiment_pH_7_4.xlsx=7.4
```

### Что означают главные поля

- `mode` - какой тип таблицы читать. Для Excel с тройками выбирайте `Excel-файлы списком file=pH`.
- `excel_files_with_ph` - список Excel-файлов и pH. Это главное поле для Excel-сценария.
- `family_rules` - как объединять образцы в отдельные картинки. Формат: `что искать в названии=название группы`.
- `default_family` - куда отправить образец, если он не подошел ни под одно правило.
- `output_dir` - папка, куда сохраняются PNG.
- `value_label`, `dose_label`, `concentration_label`, `panel_label` - подписи на графике.
- `center_colors_at_zero` - делать 0 центром цветовой шкалы. Для lnDEF обычно оставляем включенным.
- `show_numbers_in_cells` - показывать численные значения внутри клеток.

### Примеры `family_rules`

Если названия образцов такие: `NanoA 5%`, `NanoA20%`, `NanoB`, используйте:

```text
NanoA=NanoA; NanoB=NanoB
```

Если у вас клеточные линии в названиях образцов, например `HeLa NP1`, `A549 NP1`, используйте:

```text
HeLa=HeLa; A549=A549
```

Если нужно все образцы поместить в одну картинку, оставьте `family_rules` пустым и задайте `default_family`, например `All samples`.

### Как должна выглядеть Excel-таблица

Самое важное: один образец занимает три соседних столбца. В первом столбце тройки находится нужное значение `mean lnDEF`; следующие два столбца `SD` и `N` нужны для исходной таблицы, но тепловая карта их не рисует.

| Dose, Gy | NanoA 5% (0.005) | NanoA 5% (0.005) | NanoA 5% (0.005) | NanoA 5% (0.025) | NanoA 5% (0.025) | NanoA 5% (0.025) | NanoB (0.010) | NanoB (0.010) | NanoB (0.010) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
|  | mean lnDEF | SD | N | mean lnDEF | SD | N | mean lnDEF | SD | N |
| 3 | 0.22 | 0.04 | 3 | 0.60 | 0.08 | 3 | 0.10 | 0.03 | 3 |
| 6 | 0.98 | 0.10 | 3 | 1.38 | 0.12 | 3 | -0.16 | 0.05 | 3 |
| 9 | 0.16 | 0.06 | 3 | 0.96 | 0.11 | 3 | 0.27 | 0.04 | 3 |

Как скрипт читает эту таблицу:

- `Dose, Gy` становится осью X.
- Число в скобках, например `(0.005)`, становится концентрацией на оси Y.
- Текст до скобок, например `NanoA 5%`, становится названием образца.
- Из каждой тройки берется только первый столбец: `mean lnDEF`.
- pH берется не из таблицы, а из поля `excel_files_with_ph`, например `file.xlsx=6.5`.

In [ ]:
#@title Показать пример Excel-таблицы
import pandas as pd
from IPython.display import display

example_excel_table = pd.DataFrame(
    [
        ["", "mean lnDEF", "SD", "N", "mean lnDEF", "SD", "N", "mean lnDEF", "SD", "N"],
        [3, 0.22, 0.04, 3, 0.60, 0.08, 3, 0.10, 0.03, 3],
        [6, 0.98, 0.10, 3, 1.38, 0.12, 3, -0.16, 0.05, 3],
        [9, 0.16, 0.06, 3, 0.96, 0.11, 3, 0.27, 0.04, 3],
    ],
    columns=[
        "Dose, Gy",
        "NanoA 5% (0.005)", "NanoA 5% (0.005)", "NanoA 5% (0.005)",
        "NanoA 5% (0.025)", "NanoA 5% (0.025)", "NanoA 5% (0.025)",
        "NanoB (0.010)", "NanoB (0.010)", "NanoB (0.010)",
    ],
)

display(example_excel_table)
print("В каждой тройке столбцов скрипт берет 1-й столбец: mean lnDEF. SD и N игнорируются при построении heatmap.")

In [ ]:
#@title Файлы и формат данных
# 1) Для обычного Excel-сценария выберите mode = "Excel-файлы списком file=pH".
# 2) В excel_files_with_ph укажите все Excel-файлы и их pH через точку с запятой.
#    Пример: experiment_6_5.xlsx=6.5; experiment_7_4.xlsx=7.4
# 3) Если запускаете встроенный пример из репозитория, оставьте первый режим.
mode = "текущий пример: два Excel pH + ручной CSV" #@param ["текущий пример: два Excel pH + ручной CSV", "одна длинная таблица", "одна широкая таблица с дозами", "Excel-файлы списком file=pH"]

# Эти поля используются только для режима "текущий пример".
# Для нового Excel-эксперимента обычно их менять не нужно: используйте excel_files_with_ph ниже.
excel_ph_65 = "Protons BiCe GdEuF3 6_5_formatted.xlsx" #@param {type:"string"}
ph_65 = 6.5 #@param {type:"number"}
excel_ph_74 = "Protons BiCe GdEuF3 7_4_formatted.xlsx" #@param {type:"string"}
ph_74 = 7.4 #@param {type:"number"}
manual_wide_csv = "gdf3_gdeuf3_manual.csv" #@param {type:"string"}

# Эти поля используются только для режимов "одна длинная таблица" и "одна широкая таблица с дозами".
# single_table_file: имя одного CSV/XLSX файла.
# single_table_family: группа, если весь файл нужно рисовать как одну группу.
single_table_file = "" #@param {type:"string"}
single_table_family = "All samples" #@param {type:"string"}

# Главное поле для нового Excel-эксперимента.
# Формат: файл.xlsx=pH; другой_файл.xlsx=pH
# Пример: Protons BiCe GdEuF3 6_5_formatted.xlsx=6.5; Protons BiCe GdEuF3 7_4_formatted.xlsx=7.4
excel_files_with_ph = "Protons BiCe GdEuF3 6_5_formatted.xlsx=6.5; Protons BiCe GdEuF3 7_4_formatted.xlsx=7.4" #@param {type:"string"}

# Правила группировки образцов в отдельные картинки.
# Формат: текст_в_названии_образца=название_группы; следующий_текст=следующая_группа
# Пример: BiCe=BiCe; GdEu=GdF3 / GdEuF3; GdF3=GdF3 / GdEuF3
# Если оставить пустым, все образцы попадут в default_family.
family_rules = "BiCe=BiCe; GdEu=GdF3 / GdEuF3; GdF3=GdF3 / GdEuF3" #@param {type:"string"}
default_family = "All samples" #@param {type:"string"}

#@title Внешний вид графика
# output_dir: папка для PNG.
output_dir = "heatmap_outputs" #@param {type:"string"}
# value_label: подпись цветовой шкалы.
value_label = "$\\ln(\\mathrm{DEF}_{\\mathrm{ROS}})$" #@param {type:"string"}
# dose_label: подпись дозы по оси X.
dose_label = "Dose, Gy" #@param {type:"string"}
# concentration_label: подпись концентрации по оси Y.
concentration_label = "Conc, mg/mL" #@param {type:"string"}
# panel_label: подпись панелей сверху. Обычно это pH.
panel_label = "pH" #@param {type:"string"}
# colorbar_note: пояснение к цветам на шкале.
colorbar_note = "blue: ROS decrease vs control; red: ROS increase vs control" #@param {type:"string"}
# color_map: цветовая палитра. Для данных вокруг нуля удобны bwr, coolwarm, RdBu_r.
color_map = "bwr" #@param ["bwr", "coolwarm", "RdBu_r", "viridis", "magma", "plasma"]
# center_colors_at_zero: если True, 0 будет белым/центральным цветом.
center_colors_at_zero = True #@param {type:"boolean"}
# use_one_color_scale_for_all_groups: если True, разные группы можно сравнивать по цвету.
use_one_color_scale_for_all_groups = True #@param {type:"boolean"}
# show_numbers_in_cells: если True, внутри клеток будут напечатаны значения.
show_numbers_in_cells = True #@param {type:"boolean"}


In [ ]:
import importlib
import json
from pathlib import Path

if not Path("heatmap.py").exists():
    raise FileNotFoundError("Перед запуском этой ячейки загрузите heatmap.py в шаге 1.")

import heatmap
importlib.reload(heatmap)

def parse_family_rules(text):
    rules = []
    for chunk in text.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "=" not in chunk:
            raise ValueError(f"Правило группы должно выглядеть так: текст=название группы. Ошибка: {chunk}")
        contains, family = chunk.split("=", 1)
        rules.append({"contains": contains.strip(), "family": family.strip()})
    return rules

def parse_excel_file_list(text):
    sources = []
    for chunk in text.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        if "=" not in chunk:
            raise ValueError(f"Запись Excel должна выглядеть так: file.xlsx=6.5. Ошибка: {chunk}")
        path, ph = chunk.split("=", 1)
        sources.append({"path": path.strip(), "format": "triplet_excel", "ph": float(ph.strip())})
    return sources

def file_exists(path):
    return bool(path) and Path(path).exists()

sources = []
if mode == "текущий пример: два Excel pH + ручной CSV":
    if file_exists(excel_ph_65):
        sources.append({"path": excel_ph_65, "format": "triplet_excel", "ph": ph_65, "include_families": ["BiCe"]})
    if file_exists(excel_ph_74):
        sources.append({"path": excel_ph_74, "format": "triplet_excel", "ph": ph_74, "include_families": ["BiCe"]})
    if file_exists(manual_wide_csv):
        sources.append({"path": manual_wide_csv, "format": "wide_dose", "family": "GdF3 / GdEuF3"})
elif mode == "одна длинная таблица":
    sources.append({"path": single_table_file, "format": "long", "family": single_table_family})
elif mode == "одна широкая таблица с дозами":
    sources.append({"path": single_table_file, "format": "wide_dose", "family": single_table_family})
elif mode == "Excel-файлы списком file=pH":
    sources.extend(parse_excel_file_list(excel_files_with_ph))

if not sources:
    raise ValueError("Не настроен ни один источник данных. Проверьте имена загруженных файлов и выбранный режим.")

config = {
    "sources": sources,
    "family_rules": parse_family_rules(family_rules),
    "default_family": default_family,
    "plot": {
        "output_dir": output_dir,
        "value_label": value_label,
        "colorbar_note": colorbar_note,
        "dose_label": dose_label,
        "concentration_label": concentration_label,
        "panel_label": panel_label,
        "cmap": color_map,
        "center": 0 if center_colors_at_zero else None,
        "global_color_scale": use_one_color_scale_for_all_groups,
        "annotate": show_numbers_in_cells,
    },
}

print("Готовая конфигурация для запуска:\n")
print(json.dumps(config, indent=2, ensure_ascii=False))

print("\nЧто будет сделано:")
for source in sources:
    description = f"- файл: {source['path']} | формат: {source['format']}"
    if "ph" in source:
        description += f" | pH: {source['ph']}"
    if "family" in source:
        description += f" | группа: {source['family']}"
    if "include_families" in source:
        description += f" | взять только группы: {', '.join(source['include_families'])}"
    print(description)

if config["family_rules"]:
    print("\nПравила группировки:")
    for rule in config["family_rules"]:
        print(f"- если в названии образца есть '{rule['contains']}', группа будет '{rule['family']}'")
else:
    print(f"\nПравила группировки пустые: все образцы без явной группы попадут в '{default_family}'")

print(f"\nPNG будут сохранены в папку: {output_dir}")

## 3. Постройте тепловые карты

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import importlib

if "config" not in globals():
    raise RuntimeError("Сначала запустите ячейку над этой: она собирает config из блока настроек.")

if not Path("heatmap.py").exists():
    raise FileNotFoundError("Сначала запустите шаг 1: он скачивает heatmap.py из GitHub.")

import heatmap
importlib.reload(heatmap)

df = heatmap.load_dataset(config)
print(f"Загружено строк: {len(df)}")
display(df.head(20))

paths = heatmap.plot_heatmaps(df, config["plot"])
for path in paths:
    print(path)
    display(Image(filename=str(path)))


## 4. Скачайте результаты

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path(output_dir) / "heatmaps.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in paths:
        archive.write(path, arcname=Path(path).name)

print("Создан архив", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Кнопка скачивания доступна в Google Colab. При локальном запуске откройте ZIP по пути выше.")